In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import pyvista as pv
from desmin_reconstruction.plotting import save_panel
from desmin_reconstruction.plotting.overlays import compose_overlays, render_overlays
from desmin_reconstruction.plotting.renderings import (
    render_pointcloud,
)
from IPython.display import display, SVG


plt.rc("font", size=12)
plt.rc("grid", linestyle="--", color="lightgray")
plt.rc("axes.spines", right=False, top=False)
pv.set_jupyter_backend("static")

COLOR_DICT = {"desmin": "#CC3311", "actin": "#0077BB"}

datadir = Path("../../data/")

In [2]:
filepath = datadir / "STORM/desmin_actin_2.5um.csv"
df = pl.read_csv(filepath)
df = df.with_columns(*[pl.col(i) / 1000 for i in ("x", "y", "z")])
df = df.with_columns(*[pl.col(i) - pl.col(i).min() for i in ("x", "y", "z")])

df = df.filter(
    (pl.col("x") >= 9.0)
    & (pl.col("x") <= 28.0)
    & (pl.col("y") >= 9.0)
    & (pl.col("y") <= 28.0)
)
df = df.with_columns(*[pl.col(i) - pl.col(i).min() for i in ("x", "y", "z")])
out = df.partition_by("probe")

points = {
    name: df_.select(["x", "y", "z"]).to_numpy()
    for name, df_ in zip(("desmin", "actin"), out)
}

In [4]:
rendered = render_overlays(
    points,
    COLOR_DICT,
    render_pointcloud,
    protein_names=("desmin", "actin"),
    xy_bounds=None,  # Common bounds from both point clouds
    height_px=700,
    opacity=0.2,
    point_size=1.0,
)

fig, axes = compose_overlays(
    rendered,
    titles=("Desmin", "Actin", "Overlay"),
    panel_height=2.6,  # Inches
    panel_gap=0.35,  # Horizontal gap between panels, inches
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    xtick_fontsize=12,
    ytick_fontsize=12,
    title_fontsize=16,
    xtick_pad=6,  # All padding below is in points
    ytick_pad=6,
    xlabel_pad=7,  # Smaller = common X title closer to panels
    ylabel_pad=7,
    ytick_rotation=90,
    tick_format="%.1f",
)

In [5]:
savepath = "../../figures/3d"
save_panel(fig, savepath, formats=("svg",))
display(SVG(filename=str(savepath + ".svg")))
plt.close(fig)